# 04 - Validaciones y Exploración

Objetivo:
- Validar nulos en columnas esenciales de la OBT.
- Validar rangos lógicos (distancias, duraciones, montos).
- Validar coherencia temporal (pickup vs dropoff).
- Revisar duplicados por clave natural (`trip_nk`).
- Comparar conteos RAW vs OBT por servicio/año/mes.
- Construir la matriz de cobertura desde `RAW.LOAD_AUDIT`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import datetime
import glob

In [2]:
def log_step(message):
    timestamp = datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{timestamp}Z] {message}')


def build_spark(app_name):
    jars_dir = os.environ.get('SPARK_JARS_DIR', '/home/jovyan/jars')
    jar_paths = sorted(glob.glob(os.path.join(jars_dir, '*.jar')))
    builder = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.session.timeZone', 'UTC')
    )
    if jar_paths:
        jars_csv = ','.join(jar_paths)
        classpath = ':'.join(jar_paths)
        log_step(f'Loading JARs: {jar_paths}')
        builder = (
            builder
            .config('spark.jars', jars_csv)
            .config('spark.driver.extraClassPath', classpath)
            .config('spark.executor.extraClassPath', classpath)
        )
    else:
        log_step(f'No JARs found in {jars_dir}')
    return builder.getOrCreate()


def assert_snowflake_connector(spark):
    class_names = ['net.snowflake.spark.snowflake.DefaultSource', 'snowflake.DefaultSource']
    for class_name in class_names:
        try:
            spark.sparkContext._jvm.java.lang.Class.forName(class_name)
            log_step(f'Snowflake connector detected: {class_name}')
            return
        except Exception:
            continue
    raise RuntimeError('Snowflake connector not found. Run `bash scripts/download_snowflake_jars.sh` from the repo root, restart the container, and rerun this notebook.')


app = build_spark('04_validaciones_y_exploracion')
assert_snowflake_connector(app)

# Credenciales Snowflake desde variables de ambiente
sf_option = {
    'sfURL': os.environ['SF_HOST'],
    'sfUser': os.environ['SF_USER'],
    'sfPassword': os.environ['SF_PASSWORD'],
    'sfDatabase': os.environ['SF_DATABASE'],
    'sfSchema': os.environ.get('SF_RAW_SCHEMA', 'raw'),
    'sfWarehouse': os.environ.get('SF_WAREHOUSE', ''),
    'sfRole': os.environ.get('SF_ROLE', ''),
}

services = [item.strip() for item in os.environ.get('SERVICES', 'yellow,green').split(',') if item.strip()]
start_year = int(os.environ.get('START_YEAR', '2015'))
end_year = int(os.environ.get('END_YEAR', '2025'))
months = [int(item.strip()) for item in os.environ.get('MONTHS', '1,2,3,4,5,6,7,8,9,10,11,12').split(',') if item.strip()]

log_step(f'Notebook 04 ready: services={services}, years={start_year}-{end_year}, months={months}')

[2026-04-07 22:44:06Z] Loading JARs: ['/home/jovyan/jars/snowflake-jdbc-3.28.0.jar', '/home/jovyan/jars/spark-snowflake_2.12-3.1.8.jar']
[2026-04-07 22:44:10Z] Snowflake connector detected: net.snowflake.spark.snowflake.DefaultSource
[2026-04-07 22:44:10Z] Notebook 04 ready: services=['green'], years=2020-2020, months=[1]


## 1. Cargar OBT desde Snowflake

In [3]:
# Load OBT using pushdown queries to avoid Spark memory issues
# Use Snowflake SQL for heavy aggregations instead of Spark .count()

sf_utils = app._jvm.net.snowflake.spark.snowflake.Utils

def sf_run(sql):
    jsf = app._jvm.java.util.HashMap()
    for k, v in sf_option.items():
        jsf.put(k, v)
    sf_utils.runQuery(jsf, sql)

def sf_query(sql):
    return app.read.format('snowflake').options(**sf_option).option('query', sql).load()

# Get total row count via SQL (not Spark)
total_obt_rows = sf_query('SELECT COUNT(*) AS CNT FROM ANALYTICS.OBT_TRIPS').collect()[0][0]
log_step(f'ANALYTICS.OBT_TRIPS rows={total_obt_rows}')

# Load OBT for Spark operations but with lowercase columns
obt_df = (
    app.read.format('snowflake')
    .options(**sf_option)
    .option('dbtable', 'ANALYTICS.OBT_TRIPS')
    .load()
)
obt_df = obt_df.toDF(*[c.lower() for c in obt_df.columns])
obt_df.printSchema()

[2026-04-07 22:44:18Z] ANALYTICS.OBT_TRIPS rows=445964
root
 |-- obt_trip_sk: string (nullable = true)
 |-- trip_nk: string (nullable = true)
 |-- taxi_type: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- source_service: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- dropoff_date: date (nullable = true)
 |-- dropoff_hour: decimal(38,0) (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- month: decimal(38,0) (nullable = true)
 |-- year: decimal(38,0) (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- trip_year: decimal(38,0) (nullable = true)
 |-- trip_month: decimal(38,0) (nullable = true)
 |-- trip_day: decimal(38,0) (nullable = true)
 |-- trip_day_of_week: string (nullable = true)
 |-- pickup_datetime_utc: timestamp (nullable = true)
 |-- dropoff_datetime_utc: timestamp (nullable = true)
 |-- pickup_hour: deci

## 2. Validación de nulos en columnas esenciales

In [4]:
# Null validation via Snowflake SQL (much faster than Spark count on 800M rows)
null_result = sf_query("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN TRIP_NK IS NULL THEN 1 ELSE 0 END) AS null_trip_nk,
    SUM(CASE WHEN PICKUP_DATETIME_UTC IS NULL THEN 1 ELSE 0 END) AS null_pickup_datetime,
    SUM(CASE WHEN DROPOFF_DATETIME_UTC IS NULL THEN 1 ELSE 0 END) AS null_dropoff_datetime,
    SUM(CASE WHEN PULOCATIONID IS NULL THEN 1 ELSE 0 END) AS null_pulocationid,
    SUM(CASE WHEN DOLOCATIONID IS NULL THEN 1 ELSE 0 END) AS null_dolocationid,
    SUM(CASE WHEN TAXI_TYPE IS NULL THEN 1 ELSE 0 END) AS null_taxi_type,
    SUM(CASE WHEN SERVICE_TYPE IS NULL THEN 1 ELSE 0 END) AS null_service_type,
    SUM(CASE WHEN TOTAL_AMOUNT IS NULL THEN 1 ELSE 0 END) AS null_total_amount,
    SUM(CASE WHEN TRIP_DURATION_MINUTES IS NULL THEN 1 ELSE 0 END) AS null_trip_duration
FROM ANALYTICS.OBT_TRIPS
""")

print(f'Null counts (out of {total_obt_rows} rows):')
null_result.show(truncate=False)

Null counts (out of 445964 rows):
+----------+------------+--------------------+---------------------+-----------------+-----------------+--------------+-----------------+-----------------+------------------+
|TOTAL_ROWS|NULL_TRIP_NK|NULL_PICKUP_DATETIME|NULL_DROPOFF_DATETIME|NULL_PULOCATIONID|NULL_DOLOCATIONID|NULL_TAXI_TYPE|NULL_SERVICE_TYPE|NULL_TOTAL_AMOUNT|NULL_TRIP_DURATION|
+----------+------------+--------------------+---------------------+-----------------+-----------------+--------------+-----------------+-----------------+------------------+
|445964    |0           |0                   |0                    |0                |0                |0             |0                |0                |0                 |
+----------+------------+--------------------+---------------------+-----------------+-----------------+--------------+-----------------+-----------------+------------------+



## 3. Validación de rangos lógicos

In [5]:
# Range validation via Snowflake SQL
range_result = sf_query("""
SELECT 
    SUM(CASE WHEN TRIP_DISTANCE_MILES < 0 THEN 1 ELSE 0 END) AS negative_trip_distance,
    SUM(CASE WHEN TRIP_DURATION_MINUTES < 0 THEN 1 ELSE 0 END) AS negative_trip_duration,
    SUM(CASE WHEN TOTAL_AMOUNT < 0 THEN 1 ELSE 0 END) AS negative_total_amount,
    SUM(CASE WHEN FARE_AMOUNT < 0 THEN 1 ELSE 0 END) AS negative_fare_amount,
    SUM(CASE WHEN DROPOFF_DATETIME_UTC < PICKUP_DATETIME_UTC THEN 1 ELSE 0 END) AS dropoff_before_pickup,
    SUM(CASE WHEN TRIP_DISTANCE_MILES = 0 AND FARE_AMOUNT > 0 THEN 1 ELSE 0 END) AS zero_distance_positive_fare,
    SUM(CASE WHEN AVG_SPEED_MPH > 100 THEN 1 ELSE 0 END) AS speed_over_100mph,
    SUM(CASE WHEN TRIP_DURATION_MINUTES > 1440 THEN 1 ELSE 0 END) AS duration_over_24h
FROM ANALYTICS.OBT_TRIPS
""").collect()[0]

print('Range validation results:')
checks = ['negative_trip_distance', 'negative_trip_duration', 'negative_total_amount', 
           'negative_fare_amount', 'dropoff_before_pickup', 'zero_distance_positive_fare',
           'speed_over_100mph', 'duration_over_24h']
for i, check in enumerate(checks):
    count = range_result[i]
    status = 'PASS' if count == 0 else f'WARN ({count} rows)'
    print(f'  {check}: {status}')

Range validation results:
  negative_trip_distance: PASS
  negative_trip_duration: PASS
  negative_total_amount: WARN (1146 rows)
  negative_fare_amount: WARN (1146 rows)
  dropoff_before_pickup: PASS
  zero_distance_positive_fare: WARN (15021 rows)
  speed_over_100mph: WARN (584 rows)
  duration_over_24h: PASS


## 4. Detección de duplicados por clave natural

In [6]:
# Duplicate detection via Snowflake SQL
dup_result = sf_query("""
SELECT COUNT(*) AS dup_groups FROM (
    SELECT TRIP_NK, COUNT(*) AS cnt 
    FROM ANALYTICS.OBT_TRIPS 
    GROUP BY TRIP_NK 
    HAVING COUNT(*) > 1
)
""").collect()[0][0]

print(f'Duplicate trip_nk groups: {dup_result}')
if dup_result > 0:
    print('Sample duplicates:')
    sf_query("""
        SELECT TRIP_NK, COUNT(*) AS cnt 
        FROM ANALYTICS.OBT_TRIPS 
        GROUP BY TRIP_NK 
        HAVING COUNT(*) > 1 
        ORDER BY cnt DESC 
        LIMIT 10
    """).show(truncate=False)
else:
    print('No duplicates found - idempotency OK')

Duplicate trip_nk groups: 0
No duplicates found - idempotency OK


## 5. Comparación conteos RAW vs OBT por servicio/año/mes

In [7]:
# RAW vs OBT comparison via Snowflake SQL
comparison = sf_query(f"""
SELECT 
    r.SERVICE_TYPE, r.SOURCE_YEAR, r.SOURCE_MONTH,
    r.RAW_COUNT, COALESCE(o.OBT_COUNT, 0) AS OBT_COUNT,
    r.RAW_COUNT - COALESCE(o.OBT_COUNT, 0) AS DELTA,
    ROUND(COALESCE(o.OBT_COUNT, 0) / NULLIF(r.RAW_COUNT, 0) * 100, 2) AS PCT_RETAINED
FROM (
    SELECT SERVICE_TYPE, SOURCE_YEAR, SOURCE_MONTH, COUNT(*) AS RAW_COUNT
    FROM RAW.YELLOW_TRIPS GROUP BY 1,2,3
    UNION ALL
    SELECT SERVICE_TYPE, SOURCE_YEAR, SOURCE_MONTH, COUNT(*) AS RAW_COUNT
    FROM RAW.GREEN_TRIPS GROUP BY 1,2,3
) r
LEFT JOIN (
    SELECT TAXI_TYPE AS SERVICE_TYPE, SOURCE_YEAR, SOURCE_MONTH, COUNT(*) AS OBT_COUNT
    FROM ANALYTICS.OBT_TRIPS GROUP BY 1,2,3
) o ON r.SERVICE_TYPE = o.SERVICE_TYPE AND r.SOURCE_YEAR = o.SOURCE_YEAR AND r.SOURCE_MONTH = o.SOURCE_MONTH
ORDER BY r.SERVICE_TYPE, r.SOURCE_YEAR, r.SOURCE_MONTH
""")

print('RAW vs OBT row comparison:')
comparison.show(300, truncate=False)

RAW vs OBT row comparison:
+------------+-----------+------------+---------+---------+-----+------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|RAW_COUNT|OBT_COUNT|DELTA|PCT_RETAINED|
+------------+-----------+------------+---------+---------+-----+------------+
|green       |2020       |1           |446742   |445964   |778  |99.83       |
+------------+-----------+------------+---------+---------+-----+------------+



## 6. Matriz de cobertura desde LOAD_AUDIT

In [8]:
# Coverage matrix from RAW tables (LOAD_AUDIT not available)
yellow_cov = sf_query("""
SELECT 'yellow' AS SERVICE, SOURCE_YEAR, SOURCE_MONTH, 'SUCCESS' AS STATUS, COUNT(*) AS ROW_COUNT
FROM RAW.YELLOW_TRIPS GROUP BY SOURCE_YEAR, SOURCE_MONTH
""")

green_cov = sf_query("""
SELECT 'green' AS SERVICE, SOURCE_YEAR, SOURCE_MONTH, 'SUCCESS' AS STATUS, COUNT(*) AS ROW_COUNT
FROM RAW.GREEN_TRIPS GROUP BY SOURCE_YEAR, SOURCE_MONTH
""")

coverage_pdf = yellow_cov.unionByName(green_cov).toPandas()
coverage_pdf['year_month'] = coverage_pdf['SOURCE_YEAR'].astype(str) + '-' + coverage_pdf['SOURCE_MONTH'].astype(str).str.zfill(2)

print(f'Total batches loaded: {len(coverage_pdf)}')

coverage_matrix = coverage_pdf.pivot_table(
    index='year_month',
    columns='SERVICE',
    values='STATUS',
    aggfunc='last'
).fillna('MISSING')

print('Coverage matrix (2015-2025):')
coverage_matrix

Total batches loaded: 1
Coverage matrix (2015-2025):


SERVICE,green
year_month,
2020-01,SUCCESS


## 7. Resumen de conteos por servicio y exploración rápida

In [9]:
# Conteos por servicio via SQL
sf_query("SELECT TAXI_TYPE, COUNT(*) AS COUNT FROM ANALYTICS.OBT_TRIPS GROUP BY TAXI_TYPE ORDER BY TAXI_TYPE").show()

# Estadisticas descriptivas via SQL
sf_query("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT TRIP_NK) AS distinct_trip_nk,
    ROUND(AVG(TRIP_DISTANCE_MILES), 2) AS avg_distance_mi,
    ROUND(AVG(TRIP_DURATION_MINUTES), 2) AS avg_duration_min,
    ROUND(AVG(TOTAL_AMOUNT), 2) AS avg_total_amount,
    ROUND(AVG(TIP_PCT), 2) AS avg_tip_pct,
    ROUND(AVG(AVG_SPEED_MPH), 2) AS avg_speed_mph,
    SUM(CASE WHEN PU_ZONE IS NULL THEN 1 ELSE 0 END) AS null_pu_zone,
    SUM(CASE WHEN DO_ZONE IS NULL THEN 1 ELSE 0 END) AS null_do_zone
FROM ANALYTICS.OBT_TRIPS
""").show(truncate=False)

# Snapshot via SQL (just 20 rows)
sf_query("""
SELECT TRIP_NK, TAXI_TYPE, TRIP_DATE, PICKUP_HOUR, PICKUP_TIME_BAND,
    PU_BOROUGH, PU_ZONE, DO_BOROUGH, DO_ZONE,
    DISTANCE_BUCKET, TRIP_DISTANCE_MILES, TRIP_DURATION_MINUTES, AVG_SPEED_MPH,
    PAYMENT_TYPE_DESC, TOTAL_AMOUNT, TIP_PCT, ROUTE_KEY
FROM ANALYTICS.OBT_TRIPS LIMIT 20
""").show(20, truncate=False)

+---------+------+
|TAXI_TYPE| COUNT|
+---------+------+
|    green|445964|
+---------+------+

+----------+----------------+---------------+----------------+----------------+-----------+-------------+------------+------------+
|TOTAL_ROWS|DISTINCT_TRIP_NK|AVG_DISTANCE_MI|AVG_DURATION_MIN|AVG_TOTAL_AMOUNT|AVG_TIP_PCT|AVG_SPEED_MPH|NULL_PU_ZONE|NULL_DO_ZONE|
+----------+----------------+---------------+----------------+----------------+-----------+-------------+------------+------------+
|445964    |445964          |3.63           |20.14           |18.87           |8.32       |13.94        |0           |0           |
+----------+----------------+---------------+----------------+----------------+-----------+-------------+------------+------------+

+----------------------------------------------------------------+---------+----------+-----------+----------------+----------+---------------------------------+----------+----------------------------+---------------+-------------------+------